# Joining DataFrames by Columns
Often, data about the same individuals is maintained in more than one DataFrame. For example, one university office might have data about each student's time to completion of degree, while another has data about the student's tuition and financial aid.

To understand the students' experience, it may be helpful to put the two datasets together. If the data are in two DataFrames, each with one row per student, then we would want to put the columns together, making sure to match the rows so that each student's information remains on a single row.

Let us do this in the context of a simple example, and then use the method with a larger dataset.

In [1]:
import polars as pl
path_data = '../../../assets/data/'
import matplotlib
matplotlib.use('Agg')
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

The DataFrame `cones` is one we have encountered earlier. Now suppose each flavor of ice cream comes with a rating that is in a separate DataFrame.

In [2]:
cones = pl.DataFrame([
    pl.Series("Flavor", ["strawberry", "vanilla", "chocolate", "strawberry", "chocolate"]),
    pl.Series("Price", [3.55, 4.75, 6.55, 5.25, 5.75]),
])
cones

Flavor,Price
str,f64
"""strawberry""",3.55
"""vanilla""",4.75
"""chocolate""",6.55
"""strawberry""",5.25
"""chocolate""",5.75


In [3]:
ratings = pl.DataFrame([
    pl.Series("Kind", ["strawberry", "chocolate", "vanilla"]),
    pl.Series("Stars", [2.5, 3.5, 4]),
])
ratings

Kind,Stars
str,f64
"""strawberry""",2.5
"""chocolate""",3.5
"""vanilla""",4.0


Each of the DataFrames has a column that contains ice cream flavors: `cones` has the column `Flavor`, and `ratings` has the column `Kind`. The entries in these columns can be used to link the two DataFrames.

The method `join` creates a new DataFrame in which each cone in the `cones` DataFrame is augmented with the Stars information in the `ratings` DataFrame.  For each cone in `cones`, `join` finds a row in `ratings` whose `Kind` matches the cone's `Flavor`. We have to tell `join` to use those columns for matching.

In [4]:
# POLARS_CHANGE: join keeps the left row order. Sorting by Flavor puts the rows in the same order as Data 8.
rated = cones.join(ratings, left_on="Flavor", right_on="Kind").sort("Flavor")
rated

Flavor,Price,Stars
str,f64,f64
"""chocolate""",6.55,3.5
"""chocolate""",5.75,3.5
"""strawberry""",3.55,2.5
"""strawberry""",5.25,2.5
"""vanilla""",4.75,4.0


Each cone now has not only its price but also the rating of its flavor.

In general, a call to `join` that augments a DataFrame (say `df1`) with information from another DataFrame (say `df2`) looks like this:

    df1.join(df2, left_on=df1_column_for_joining, right_on=df2_column_for_joining)

The new DataFrame `rated` allows us to work out the price per star, which you can think of as an informal measure of value. Low values are good – they mean that you are paying less for each rating star.

In [5]:
rated.with_columns(
    (pl.col("Price") / pl.col("Stars")).alias("$/Star")
).sort(pl.nth(3))

Flavor,Price,Stars,$/Star
str,f64,f64,f64
"""vanilla""",4.75,4.0,1.1875
"""strawberry""",3.55,2.5,1.42
"""chocolate""",5.75,3.5,1.642857
"""chocolate""",6.55,3.5,1.871429
"""strawberry""",5.25,2.5,2.1


Though strawberry has the lowest rating among the three flavors, the less expensive strawberry cone does well on this measure because it doesn't cost a lot per star.

**Side note.** Does the order we list the two DataFrames matter? Let's try it.  As you see it, this changes the order that the columns appear in, and can potentially changes the order of the rows, but it doesn't make any fundamental difference.

In [6]:
ratings.join(cones, left_on="Kind", right_on="Flavor").sort("Kind")

Kind,Stars,Price
str,f64,f64
"""chocolate""",3.5,6.55
"""chocolate""",3.5,5.75
"""strawberry""",2.5,3.55
"""strawberry""",2.5,5.25
"""vanilla""",4.0,4.75


Also note that the join will only contain information about items that appear in both DataFrames. Let's see an example. Suppose there is a DataFrame of reviews of some ice cream cones, and we have found the average review for each flavor.

In [7]:
reviews = pl.DataFrame([
    pl.Series("Flavor", ["vanilla", "chocolate", "vanilla", "chocolate"]),
    pl.Series("Stars", [5, 3, 5, 4]),
])
reviews

Flavor,Stars
str,i64
"""vanilla""",5
"""chocolate""",3
"""vanilla""",5
"""chocolate""",4


In [8]:
average_review = reviews.group_by("Flavor").mean().sort("Flavor")
average_review

Flavor,Stars
str,f64
"""chocolate""",3.5
"""vanilla""",5.0


We can join `cones` and `average_review` by providing the labels of the columns by which to join.

In [9]:
cones.join(average_review, on="Flavor").sort("Flavor")

Flavor,Price,Stars
str,f64,f64
"""chocolate""",6.55,3.5
"""chocolate""",5.75,3.5
"""vanilla""",4.75,5.0


Notice how the strawberry cones have disappeared. None of the reviews are for strawberry cones, so there is nothing to which the `strawberry` rows can be joined. This might be a problem, or it might not be - that depends on the analysis we are trying to perform with the joined DataFrame.